In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn.functional as fun

from torchvision import datasets, transforms
from matplotlib import pyplot as plt

import import_ipynb
from common import test_model_sgd_2 # type: ignore
from per_lin_softmax_ce import Per_Lin_Softmax_CE_Autograd, \
                               Per_Lin_Softmax_CE_Backward, \
                               Per_Lin_Softmax_CE_Gradient # type: ignore

In [ ]:
#
# Load the dataset and show the first 10 samples.
#

to_tensor = transforms.ToTensor()
dataset = datasets.MNIST(root='data', train=True, download=True, transform=to_tensor)

for i in range(10):
  plt.imshow(dataset.data[i], cmap='gray')
  plt.title(f'Label: {dataset.targets[i]}')
  plt.show()

In [ ]:
#
# Train model
#
S=10000

samples_train = dataset.data[:S].float() / 255.0
samples_train = tr.reshape(samples_train, (samples_train.shape[0], -1))

labels_train = dataset.targets[:S]
labels_train = tr.reshape(labels_train, (-1, 1))

# (_, model_W, _) = logistic_regression_nc(samples_train, labels_train, epochs=1000, lr=0.01)
# assert_eq(model_W.T.shape, (10, 28*28))

In [ ]:
def onehot(z: tr.Tensor) -> tr.Tensor:
    S, C = z.shape
    p = tr.softmax(z, dim=1)
    idx = p.argmax(dim=1)
    t = tr.zeros_like(p)
    t[tr.arange(S), idx] = 1.0

In [ ]:
x = samples_train
y = fun.one_hot(labels_train.squeeze().long(), num_classes=10).float()

model = Per_Lin_Softmax_CE_Autograd(in_features=28*28, out_features=10)

test_model_sgd_2(model, x, y, epochs=1000, lr=0.01)

In [ ]:
tr.isclose(model.predict(x), y, atol=0.01, rtol=0.0).float().mean().item()

In [ ]:
#
# Show the weights as images.
#

for i in range(10):
  weights = model.linear.weight[i].reshape(28, 28).detach().numpy()

  plt.imshow(weights, cmap='gray')
  plt.title(f"Digit {i} weights")
  plt.axis('off')
  plt.show()
